In [1]:
import torch
import os
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")


In [ ]:
LOL_Directory = "/Users/"
Dataset_csv = "league_of_legends_data_large.csv"

Final_Dataset = os.path.join(LOL_Directory,Dataset_csv)

dt = pd.read_csv("Final_Dataset")


In [3]:
display(dt)

,win,kills,deaths,assists,gold_earned,cs,wards_placed,wards_killed,damage_dealt
0,0,16,6,19,17088,231,11,7,15367
1,1,8,8,5,14865,259,10,2,38332
2,0,0,17,11,15919,169,14,5,24642
3,0,19,11,1,11534,264,14,3,15789
4,0,12,7,6,18926,124,15,7,40268
...,...,...,...,...,...,...,...,...,...
995,0,2,15,12,17170,294,8,6,33469
996,0,5,13,4,19524,236,14,3,8845
997,1,8,7,8,7961,139,11,7,49650
998,1,5,17,5,8226,193,9,9,28290


In [4]:
# Separate features and targets
X = dt.drop('win', axis=1)
y = dt['win']

# print the first few rows of the data
display(X.head())

# print the first few rows of the target
display(y.head())

,kills,deaths,assists,gold_earned,cs,wards_placed,wards_killed,damage_dealt
0,16,6,19,17088,231,11,7,15367
1,8,8,5,14865,259,10,2,38332
2,0,17,11,15919,169,14,5,24642
3,19,11,1,11534,264,14,3,15789
4,12,7,6,18926,124,15,7,40268


0    0
1    1
2    0
3    0
4    0
Name: win, dtype: int64

In [5]:
display(X)
display(y)

,kills,deaths,assists,gold_earned,cs,wards_placed,wards_killed,damage_dealt
0,16,6,19,17088,231,11,7,15367
1,8,8,5,14865,259,10,2,38332
2,0,17,11,15919,169,14,5,24642
3,19,11,1,11534,264,14,3,15789
4,12,7,6,18926,124,15,7,40268
...,...,...,...,...,...,...,...,...
995,2,15,12,17170,294,8,6,33469
996,5,13,4,19524,236,14,3,8845
997,8,7,8,7961,139,11,7,49650
998,5,17,5,8226,193,9,9,28290


0      0
1      1
2      0
3      0
4      0
      ..
995    0
996    0
997    1
998    1
999    0
Name: win, Length: 1000, dtype: int64

In [6]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

display(f'X_train shape: {X_train.shape}')
display(f'y_train shape: {y_train.shape}')
display(f'X_test shape: {X_test.shape}')
display(f'y_test shape: {y_test.shape}')

'X_train shape: (800, 8)'

'y_train shape: (800,)'

'X_test shape: (200, 8)'

'y_test shape: (200,)'

In [7]:
# Standardize the data
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform it
X_train = scaler.fit_transform(X_train)

# Transform the test data using the same scaler
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) # unsqueeze for binary classification
y_test = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1) 

# Convert scaled NumPy arrays to PyTorch tensors
#X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
#y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) # unsqueeze for binary classification
#X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
#y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1) 

# Create DataLoader for training and test sets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [8]:
display(f'X_train shape: {X_train.shape}')
display(f'y_train shape: {y_train.shape}')
display(f'X_test shape: {X_test.shape}')
display(f'y_test shape: {y_test.shape}')

'X_train shape: torch.Size([800, 8])'

'y_train shape: torch.Size([800, 1])'

'X_test shape: torch.Size([200, 8])'

'y_test shape: torch.Size([200, 1])'

In [9]:
# Create Logestic_Regression class
class Logestic_Regression(nn.Module):
    # Constructor
    def __init__(self, n_input):
        super(Logestic_Regression, self).__init__()
        self.linear = nn.Linear(n_input,1)

    # Prediction    
    def forward(self, x):
        yhat = torch.sigmoid(self.linear(x))
        return yhat


In [10]:
input_dim = X_train.shape[1]

In [11]:
# Instantiate the model
model = Logestic_Regression(input_dim)

In [12]:
print(model)

Logestic_Regression(
  (linear): Linear(in_features=8, out_features=1, bias=True)
)


In [13]:
# Define the loss function and optimizer
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001) 

#Display the Parameters
model.state_dict()

OrderedDict([('linear.weight',
              tensor([[ 0.0657, -0.2137, -0.2323, -0.1348, -0.1493,  0.2811, -0.3132,  0.2458]])),
             ('linear.bias', tensor([0.0386]))])

In [14]:
epochs = 1000

In [15]:
train_losses = []
test_losses = []

In [16]:
for epoch in range(epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        train_outputs = model(X_batch)
        loss = criterion(train_outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    if (epoch + 1) % 100 == 0:
        print(f'Epoch for Train: [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

for epoch in range(epochs):
    # Evaluation phase on test set
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            test_outputs = model(X_batch)
            loss = criterion(test_outputs, y_batch)
            test_loss += loss.item()

    test_loss /= len(test_loader)
    test_losses.append(test_loss)
    
    if (epoch + 1) % 100 == 0:
        print(f'Epoch for Test: [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


Epoch for Train: [100/1000], Loss: 0.6738
Epoch for Train: [200/1000], Loss: 0.5836
Epoch for Train: [300/1000], Loss: 0.6719
Epoch for Train: [400/1000], Loss: 0.5784
Epoch for Train: [500/1000], Loss: 0.7006
Epoch for Train: [600/1000], Loss: 0.6039
Epoch for Train: [700/1000], Loss: 0.7386
Epoch for Train: [800/1000], Loss: 0.6084
Epoch for Train: [900/1000], Loss: 0.7084
Epoch for Train: [1000/1000], Loss: 0.7045
Epoch for Test: [100/1000], Loss: 0.5732
Epoch for Test: [200/1000], Loss: 0.5732
Epoch for Test: [300/1000], Loss: 0.5732
Epoch for Test: [400/1000], Loss: 0.5732
Epoch for Test: [500/1000], Loss: 0.5732
Epoch for Test: [600/1000], Loss: 0.5732
Epoch for Test: [700/1000], Loss: 0.5732
Epoch for Test: [800/1000], Loss: 0.5732
Epoch for Test: [900/1000], Loss: 0.5732
Epoch for Test: [1000/1000], Loss: 0.5732


In [17]:
# Calculate Accuracy for training set
train_predicted = (train_outputs >= 0.5).float()
train_correct = (train_predicted == y_train).sum().item()
train_accuracy = train_correct / y_train.size(0)

# Calculate Accuracy for test set
test_predicted = (test_outputs >= 0.5).float()
test_correct = (test_predicted == y_test).sum().item()
test_accuracy = test_correct / y_test.size(0)

# Print Accuracy
print(f'Training Accuracy: {train_accuracy:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

Training Accuracy: 0.4900
Test Accuracy: 0.5100


In [18]:
# Example raw samples from the dataset (rows 0–4, only features)
# order: kills, deaths, assists, gold_earned, cs, wards_placed, wards_killed, damage_dealt
test_samples_np = np.array([
    [16,  6, 19, 17088, 231, 11, 7, 15367],  # win = 0
    [8,   8,  5, 14865, 259, 10, 2, 38332],  # win = 1
    [0,  17, 11, 15919, 169, 14, 5, 24642],  # win = 0
    [19, 11,  1, 11534, 264, 14, 3, 15789],  # win = 0
    [12,  7,  6, 18926, 124, 15, 7, 40268],  # win = 0
], dtype=np.float32)


In [19]:
# Scale using the SAME scaler used during training
test_samples_scaled = scaler.transform(test_samples_np)

# Convert to torch tensor
test_samples_t = torch.tensor(test_samples_scaled, dtype=torch.float32)

# Put model in eval mode and disable gradients
model.eval()
with torch.no_grad():
    logits = model(test_samples_t)           # shape: [5, 1] for binary classifier
    probs = torch.sigmoid(logits)           # convert to probabilities
    preds = (probs >= 0.5).long()           # threshold at 0.5

print("Probabilities:\n", probs.squeeze().cpu().numpy())
print("Predicted win labels:\n", preds.squeeze().cpu().numpy())


Probabilities:
 [0.63752574 0.621903   0.62589896 0.63589853 0.6318902 ]
Predicted win labels:
 [1 1 1 1 1]


In [20]:
true_labels = np.array([0, 1, 0, 0, 0], dtype=np.int64)  # from CSV rows 0–4
print("True labels:      ", true_labels)

True labels:       [0 1 0 0 0]
